# Agentic RAG

Retrieval-augmented generation (RAG) answers a question by first retrieving
relevant text and then asking the model to answer using that text, instead of
relying only on what the model already knows. "Agentic" RAG adds a decision step:
the graph checks whether the retrieved text actually answers the question, and if
not, it rewrites the question and tries again instead of answering anyway. This
notebook builds that loop over a small set of made-up product documents, and asks
one question the documents answer and one they do not.

In [ ]:
from typing import Annotated, Literal, TypedDict

from dotenv import load_dotenv
from pydantic import BaseModel

from langchain_core.documents import Document
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools.retriever import create_retriever_tool
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

In [ ]:
load_dotenv()

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

## Part 1: A Small Document Set

These are short made-up paragraphs about a fictional product, the "Nimbus Home
Hub". No downloads are needed; the text lives directly in this notebook.

In [ ]:
DOCS = [
    "The Nimbus Home Hub is a smart home hub that connects lights, locks, cameras, "
    "and thermostats over WiFi and Zigbee, and controls them from one app.",

    "To set up the Nimbus Home Hub, plug it into power, open the Nimbus app, and "
    "follow the in-app pairing steps. The hub needs a 2.4GHz WiFi network during setup.",

    "The Nimbus Home Hub supports integrations with Philips Hue lights, August "
    "smart locks, Ecobee thermostats, and most Zigbee-certified cameras.",

    "The Nimbus Home Hub is free to use with local control. An optional Nimbus+ "
    "subscription adds 30 days of cloud video recording and remote access when away from home.",

    "The Nimbus Home Hub is powered by a USB-C adapter and includes a 10-foot cable. "
    "It has no internal battery and must stay plugged in to work.",

    "The Nimbus Home Hub comes with a 2-year limited warranty. Customers must "
    "register their hub in the Nimbus app within 30 days of purchase to activate the warranty.",

    "If the Nimbus Home Hub will not connect to WiFi, hold the reset button on the "
    "back for 10 seconds and repeat setup. Most connection issues are fixed by moving the hub closer to the router.",

    "By default, the Nimbus Home Hub processes device automations locally on the "
    "hub itself. Only Nimbus+ subscribers' video clips are uploaded to the cloud, and those are kept for 30 days.",
]

## Part 2: Embedding the Documents

The documents are embedded once into an `InMemoryVectorStore`. This is the only
place embeddings are computed; the retriever reuses this same store for every
question below.

In [ ]:
documents = [Document(page_content=text) for text in DOCS]
vectorstore = InMemoryVectorStore.from_documents(documents, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

## Part 3: The Retriever as a Tool

`create_retriever_tool` wraps a retriever as a normal tool, with a name and
description the model uses to decide when to call it. This is the same tool-calling
mechanism as `15_Tools` and `16_Prebuilt_Agents`; the only difference is that the
tool's job is to fetch text instead of doing a calculation or a search.

In [ ]:
retriever_tool = create_retriever_tool(
    retriever,
    name="search_nimbus_docs",
    description="Search the Nimbus Home Hub product documentation for relevant passages.",
)
tools = [retriever_tool]
llm_with_tools = llm.bind_tools(tools)

## Part 4: Graph State

Besides `messages`, the state tracks `rewrite_count` and the most recent grading
result, so the routing step after grading can decide what to do next.

In [ ]:
class RAGState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    rewrite_count: int
    grade: str


MAX_REWRITES = 2

## Part 5: The Agent Node

The agent node is the same shape as the chat node in `15_Tools/Tooling.ipynb`: it
sends the conversation to a tool-bound model and returns whatever the model
produces, a tool call or a direct answer.

In [ ]:
RAG_SYSTEM_PROMPT = (
    "You answer questions about the Nimbus Home Hub. Always use search_nimbus_docs "
    "before answering. If the search results do not contain the answer, say so plainly "
    "instead of guessing."
)


def agent_node(state: RAGState) -> dict:
    response = llm_with_tools.invoke([SystemMessage(content=RAG_SYSTEM_PROMPT)] + state["messages"])
    return {"messages": [response]}

## Part 6: Grading the Retrieved Documents

After the retriever tool runs, `grade_documents` asks the model a yes/no question
with structured output: do these retrieved passages actually answer the original
question? The result is stored in state as `"grade"` instead of being routed on
directly, so the routing function in the next part stays a plain, easy-to-read
decision.

In [ ]:
class Grade(BaseModel):
    binary_score: Literal["no", "yes"]


def grade_documents(state: RAGState) -> dict:
    question = next(m.content for m in reversed(state["messages"]) if isinstance(m, HumanMessage))
    retrieved = state["messages"][-1].content

    prompt = (
        f"Question: {question}\n"
        f"Retrieved text: {retrieved}\n"
        "Does the retrieved text answer the question? Answer yes or no."
    )
    result = llm.with_structured_output(Grade).invoke(prompt)
    return {"grade": result.binary_score}

## Part 7: Generate or Rewrite

If the grade is `"yes"`, or the rewrite limit has been reached, the graph generates
a final answer from the retrieved text. Otherwise it rewrites the question and lets
the agent node try retrieval again.

In [ ]:
def route_after_grade(state: RAGState) -> Literal["generate", "rewrite"]:
    if state["grade"] == "yes":
        return "generate"
    if state.get("rewrite_count", 0) >= MAX_REWRITES:
        return "generate"
    return "rewrite"

In [ ]:
def rewrite_node(state: RAGState) -> dict:
    question = next(m.content for m in reversed(state["messages"]) if isinstance(m, HumanMessage))
    prompt = (
        f"Rewrite this question to make it easier to find in product documentation, "
        f"keeping the same meaning: {question}"
    )
    rewritten = llm.invoke(prompt)
    return {
        "messages": [HumanMessage(content=rewritten.content)],
        "rewrite_count": state.get("rewrite_count", 0) + 1,
    }

In [ ]:
def generate_node(state: RAGState) -> dict:
    question = next(m.content for m in reversed(state["messages"]) if isinstance(m, HumanMessage))
    retrieved = state["messages"][-1].content
    prompt = (
        f"Question: {question}\n"
        f"Retrieved text: {retrieved}\n"
        "Answer the question using only the retrieved text. If it does not contain "
        "the answer, say the documentation does not cover this."
    )
    answer = llm.invoke(prompt)
    return {"messages": [AIMessage(content=answer.content)]}

## Part 8: Building the Graph

```text
START -> agent -> tools_condition -> retrieve -> grade_documents
                                                        |
                                          +-------------+-------------+
                                          |                           |
                                       generate                    rewrite
                                          |                           |
                                         END                       agent (loop)
```

In [ ]:
graph = StateGraph(RAGState)
graph.add_node("agent", agent_node)
graph.add_node("retrieve", ToolNode(tools))
graph.add_node("grade_documents", grade_documents)
graph.add_node("generate", generate_node)
graph.add_node("rewrite", rewrite_node)

graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", tools_condition, {"tools": "retrieve", END: END})
graph.add_edge("retrieve", "grade_documents")
graph.add_conditional_edges("grade_documents", route_after_grade, {"generate": "generate", "rewrite": "rewrite"})
graph.add_edge("rewrite", "agent")
graph.add_edge("generate", END)

app = graph.compile()

## Part 9: A Question the Docs Answer

This asks about something covered directly in `DOCS`.

In [ ]:
result = app.invoke({
    "messages": [HumanMessage(content="What is the warranty period for the Nimbus Home Hub?")],
    "rewrite_count": 0,
})
print(result["messages"][-1].content)

## Part 10: A Question the Docs Do Not Answer

This asks about something the documents never mention. A good run either says the
documentation does not cover it, or reaches the rewrite limit and says so, instead
of inventing an answer.

In [ ]:
result = app.invoke({
    "messages": [HumanMessage(content="What programming language does the Nimbus Home Hub's firmware use?")],
    "rewrite_count": 0,
})
print(result["messages"][-1].content)

## Summary

Plain RAG retrieves text and answers from it. Agentic RAG adds a check in the
middle: `grade_documents` decides whether the retrieved text is actually useful
before the graph commits to an answer. When it is not, `rewrite_node` gives the
retriever a better-worded question and the graph tries again, through the same
`agent -> tools_condition -> retrieve` path as the first attempt. `MAX_REWRITES`
keeps that loop from running forever when no rewrite will help, the same kind of
guard used for the supervisor's step limit in `20_Multi_Agent`.